# Image Quality Classifier (Good/Fair/Bad/Worst)

This notebook trains a model to classify ROP fundus images into quality categories:
- **Good**: Images suitable for diagnosis
- **Fair**: Lower quality but usable images
- **Bad**: Low quality images unsuitable for diagnosis
- **Worst**: Very poor quality images (e.g., completely out of focus, no retinal structure visible)

## Objective
Create a pre-inference screening model to filter out low-quality images before ROP classification.
Images classified as **Bad** or **Worst** are excluded from ROP classifier training.

## Data
- Kubota_selection folder with manually labeled quality categories
- Good / Fair / Bad / Worst

In [ ]:
# ==================== Environment Setup ====================
import os
import sys
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Any
import warnings
warnings.filterwarnings('ignore')

# Core libraries
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

# Image processing
import cv2
from PIL import Image

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR

# timm for EfficientNet
import timm

# Metrics
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    confusion_matrix, classification_report, accuracy_score,
    f1_score, precision_score, recall_score
)

# Albumentations for augmentation
import albumentations as A
from albumentations.pytorch import ToTensorV2

from datetime import datetime

# Set seeds for reproducibility
def set_seed(seed: int = 42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ==================== Configuration ====================

class QualityConfig:
    """Configuration for quality classifier training."""

    # Paths
    DATA_ROOT = Path(r"E:\Multicenter_ROP_study")
    KUBOTA_DIR = DATA_ROOT / "Multicenter_images" / "Kubota_selection"
    OUTPUT_DIR = Path(r"C:\Users\ykita\ROP_AI_project\ROP_project\multicenter_study\outputs_quality_classifier_4class")

    # Model
    MODEL_NAME = "efficientnet_b0"
    PRETRAINED = True
    NUM_CLASSES = 4  # Good, Fair, Bad, Worst
    DROPOUT = 0.3

    # Image
    IMG_SIZE = 512

    # Training
    BATCH_SIZE = 16
    NUM_WORKERS = 0
    EPOCHS = 30
    LEARNING_RATE = 1e-4
    WEIGHT_DECAY = 1e-4
    PATIENCE = 5

    # Cross-validation
    N_FOLDS = 5

    # Label smoothing
    LABEL_SMOOTHING = 0.1

    # Class names
    CLASS_NAMES = ["Good", "Fair", "Bad", "Worst"]
    CLASS_TO_IDX = {"Good": 0, "Fair": 1, "Bad": 2, "Worst": 3}

    # Version info
    VERSION = "v2"
    DESCRIPTION = "4-class quality classifier (Good/Fair/Bad/Worst)"

config = QualityConfig()

# Create output directory
config.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output directory: {config.OUTPUT_DIR}")

## 1. Data Loading

In [ ]:
# ==================== Load Quality-Labeled Images ====================

def load_quality_dataset(kubota_dir: Path) -> pd.DataFrame:
    """
    Load images from Kubota_selection folder with quality labels.
    
    Folder structure:
    Kubota_selection/
    ├── Good/
    ├── Fair/
    ├── Bad/
    └── Worst/
    """
    data = []
    
    for quality in ["Good", "Fair", "Bad", "Worst"]:
        folder = kubota_dir / quality
        if not folder.exists():
            print(f"Warning: Folder not found: {folder}")
            continue
        
        for img_path in folder.glob("*.png"):
            # Skip empty files
            if img_path.stat().st_size == 0:
                continue
            
            # Extract video_id from filename (e.g., "0001_AMU_00110.png" -> "0001_AMU")
            filename = img_path.stem
            parts = filename.rsplit('_', 1)
            video_id = parts[0] if len(parts) >= 2 else filename
            
            data.append({
                "image_path": str(img_path),
                "image_name": img_path.name,
                "video_id": video_id,
                "quality": quality,
                "quality_label": QualityConfig.CLASS_TO_IDX[quality]
            })
    
    df = pd.DataFrame(data)
    
    # Summary
    print("=== Quality Distribution ===")
    print(df['quality'].value_counts())
    print(f"\nTotal images: {len(df)}")
    
    return df

# Load dataset
dataset_df = load_quality_dataset(config.KUBOTA_DIR)
display(dataset_df.head())

In [ ]:
# ==================== Class Weights for Imbalance ====================

def compute_class_weights(df: pd.DataFrame) -> torch.Tensor:
    """Compute class weights for handling class imbalance."""
    class_counts = df['quality_label'].value_counts().sort_index()
    total = len(df)
    
    # Inverse frequency weighting
    weights = total / (len(class_counts) * class_counts.values)
    
    print("Class weights:")
    for i, (cls, weight) in enumerate(zip(config.CLASS_NAMES, weights)):
        print(f"  {cls}: {weight:.4f}")
    
    return torch.FloatTensor(weights)

class_weights = compute_class_weights(dataset_df)

## 2. Dataset and Augmentation

In [ ]:
# ==================== Data Augmentation ====================

def get_train_transforms(img_size: int = 512) -> A.Compose:
    """Training augmentation pipeline."""
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Rotate(limit=180, p=0.8, border_mode=cv2.BORDER_CONSTANT, value=0),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.OneOf([
            A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=1.0),
            A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=1.0),
        ], p=0.5),
        A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=0.3),
        A.OneOf([
            A.GaussianBlur(blur_limit=(3, 5), p=1.0),
            A.MedianBlur(blur_limit=5, p=1.0),
        ], p=0.3),
        A.GaussNoise(var_limit=(10.0, 30.0), p=0.2),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])

def get_valid_transforms(img_size: int = 512) -> A.Compose:
    """Validation transforms (no augmentation)."""
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])

train_transforms = get_train_transforms(config.IMG_SIZE)
valid_transforms = get_valid_transforms(config.IMG_SIZE)
print("Transforms defined successfully")

In [ ]:
# ==================== Dataset Class ====================

class QualityDataset(Dataset):
    """PyTorch Dataset for image quality classification."""
    
    def __init__(self, df: pd.DataFrame, transforms: A.Compose = None):
        self.df = df.reset_index(drop=True)
        self.transforms = transforms
    
    def __len__(self) -> int:
        return len(self.df)
    
    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        row = self.df.iloc[idx]
        
        image = cv2.imread(row['image_path'])
        if image is None:
            raise ValueError(f"Failed to load image: {row['image_path']}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        if self.transforms:
            transformed = self.transforms(image=image)
            image = transformed['image']
        
        label = torch.tensor(row['quality_label'], dtype=torch.long)
        
        return {
            'image': image,
            'label': label,
            'image_path': row['image_path']
        }

# Test dataset
test_dataset = QualityDataset(dataset_df.head(5), transforms=train_transforms)
sample = test_dataset[0]
print(f"Sample image shape: {sample['image'].shape}")
print(f"Sample label: {sample['label']}")

## 3. Model Architecture

In [ ]:
# ==================== Quality Classifier Model ====================

class QualityClassifier(nn.Module):
    """EfficientNet-based quality classifier."""
    
    def __init__(self, model_name: str = "efficientnet_b0", num_classes: int = 3, 
                 pretrained: bool = True, dropout: float = 0.3):
        super().__init__()
        
        self.backbone = timm.create_model(model_name, pretrained=pretrained, num_classes=0, global_pool='avg')
        in_features = self.backbone.num_features
        
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(dropout / 2),
            nn.Linear(256, num_classes)
        )
        
        print(f"Model: {model_name}, Features: {in_features}, Classes: {num_classes}")
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        features = self.backbone(x)
        return self.classifier(features)

# Test model
model = QualityClassifier(model_name=config.MODEL_NAME, num_classes=config.NUM_CLASSES)
model = model.to(device)

with torch.no_grad():
    dummy_input = torch.randn(2, 3, config.IMG_SIZE, config.IMG_SIZE).to(device)
    output = model(dummy_input)
    print(f"Output shape: {output.shape}")

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

## 4. Training Functions

In [ ]:
# ==================== Training Loop ====================

def train_epoch(model, dataloader, criterion, optimizer, scheduler, device):
    """Train for one epoch."""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for batch in tqdm(dataloader, desc="Training", leave=False):
        images = batch['image'].to(device)
        labels = batch['label'].to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    return running_loss / len(dataloader), correct / total


def validate_epoch(model, dataloader, criterion, device):
    """Validate for one epoch."""
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    all_probs = []
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Validation", leave=False):
            images = batch['image'].to(device)
            labels = batch['label'].to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item()
            
            probs = F.softmax(outputs, dim=1)
            _, predicted = outputs.max(1)
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_probs = np.array(all_probs)
    
    metrics = {
        'accuracy': accuracy_score(all_labels, all_preds),
        'f1_macro': f1_score(all_labels, all_preds, average='macro'),
        'f1_weighted': f1_score(all_labels, all_preds, average='weighted'),
        'precision_macro': precision_score(all_labels, all_preds, average='macro'),
        'recall_macro': recall_score(all_labels, all_preds, average='macro'),
    }
    
    return running_loss / len(dataloader), metrics, all_preds, all_labels, all_probs

In [ ]:
# ==================== Cross-Validation ====================

def run_cross_validation(df: pd.DataFrame, config: QualityConfig) -> Dict:
    """Run 5-fold cross-validation."""
    
    # StratifiedGroupKFold (patient-level split)
    sgkf = StratifiedGroupKFold(n_splits=config.N_FOLDS, shuffle=True, random_state=42)
    
    all_fold_results = []
    all_predictions = []
    
    for fold, (train_idx, val_idx) in enumerate(sgkf.split(df, df['quality_label'], df['video_id'])):
        print(f"\n{'='*50}")
        print(f"Fold {fold + 1}/{config.N_FOLDS}")
        print(f"{'='*50}")
        
        fold_dir = config.OUTPUT_DIR / f"fold_{fold + 1}"
        fold_dir.mkdir(exist_ok=True)
        
        # Split data
        train_df = df.iloc[train_idx].reset_index(drop=True)
        val_df = df.iloc[val_idx].reset_index(drop=True)
        
        print(f"Train: {len(train_df)}, Val: {len(val_df)}")
        print(f"Train quality distribution: {train_df['quality'].value_counts().to_dict()}")
        print(f"Val quality distribution: {val_df['quality'].value_counts().to_dict()}")
        
        # Create datasets
        train_dataset = QualityDataset(train_df, transforms=train_transforms)
        val_dataset = QualityDataset(val_df, transforms=valid_transforms)
        
        # Weighted sampler for class imbalance
        train_labels = train_df['quality_label'].values
        class_counts = np.bincount(train_labels, minlength=config.NUM_CLASSES)
        class_weights_fold = 1.0 / np.maximum(class_counts, 1)
        sample_weights = class_weights_fold[train_labels]
        sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)
        
        train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE, sampler=sampler, num_workers=config.NUM_WORKERS)
        val_loader = DataLoader(val_dataset, batch_size=config.BATCH_SIZE, shuffle=False, num_workers=config.NUM_WORKERS)
        
        # Initialize model
        model = QualityClassifier(model_name=config.MODEL_NAME, num_classes=config.NUM_CLASSES, 
                                  pretrained=config.PRETRAINED, dropout=config.DROPOUT)
        model = model.to(device)
        
        # Loss with class weights
        criterion = nn.CrossEntropyLoss(weight=class_weights.to(device), label_smoothing=config.LABEL_SMOOTHING)
        optimizer = AdamW(model.parameters(), lr=config.LEARNING_RATE, weight_decay=config.WEIGHT_DECAY)
        scheduler = OneCycleLR(optimizer, max_lr=config.LEARNING_RATE, epochs=config.EPOCHS, steps_per_epoch=len(train_loader))
        
        # Training loop
        best_val_loss = float('inf')
        patience_counter = 0
        train_losses = []
        val_losses = []
        train_accs = []
        val_accs = []
        
        for epoch in range(config.EPOCHS):
            train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, scheduler, device)
            val_loss, metrics, _, _, _ = validate_epoch(model, val_loader, criterion, device)
            
            train_losses.append(train_loss)
            val_losses.append(val_loss)
            train_accs.append(train_acc)
            val_accs.append(metrics['accuracy'])
            
            print(f"Epoch {epoch + 1}: Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, Val Acc: {metrics['accuracy']:.4f}, Val F1: {metrics['f1_macro']:.4f}")
            
            # Early stopping
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                patience_counter = 0
                torch.save(model.state_dict(), fold_dir / "best_model.pt")
            else:
                patience_counter += 1
                if patience_counter >= config.PATIENCE:
                    print(f"Early stopping at epoch {epoch + 1}")
                    break
        
        # Load best model and evaluate
        model.load_state_dict(torch.load(fold_dir / "best_model.pt"))
        _, final_metrics, preds, labels, probs = validate_epoch(model, val_loader, criterion, device)
        
        # Save fold results
        fold_result = {
            'fold': fold + 1,
            'train_size': len(train_df),
            'val_size': len(val_df),
            'best_val_loss': best_val_loss,
            'metrics': final_metrics,
            'train_losses': train_losses,
            'val_losses': val_losses,
            'train_accs': train_accs,
            'val_accs': val_accs
        }
        all_fold_results.append(fold_result)
        
        # Save predictions
        for i, row in val_df.iterrows():
            idx = val_df.index.get_loc(i)
            pred_entry = {
                'fold': fold + 1,
                'image_path': row['image_path'],
                'video_id': row['video_id'],
                'true_label': labels[idx],
                'true_quality': row['quality'],
                'pred_label': preds[idx],
                'pred_quality': config.CLASS_NAMES[preds[idx]],
            }
            for cls_idx, cls_name in enumerate(config.CLASS_NAMES):
                pred_entry[f'prob_{cls_name.lower()}'] = probs[idx][cls_idx]
            all_predictions.append(pred_entry)
        
        # Cleanup
        del model, optimizer, scheduler
        torch.cuda.empty_cache()
    
    # Save predictions
    predictions_df = pd.DataFrame(all_predictions)
    predictions_df.to_csv(config.OUTPUT_DIR / "predictions.csv", index=False)
    
    return {
        'fold_results': all_fold_results,
        'predictions_df': predictions_df
    }

## 5. Run Training

In [ ]:
# ==================== Run Cross-Validation ====================

results = run_cross_validation(dataset_df, config)

## 6. Results Analysis

In [ ]:
# ==================== Aggregate Results ====================

def aggregate_results(fold_results: List[Dict]) -> Dict[str, Tuple[float, float]]:
    """Aggregate metrics across folds."""
    metrics = {}
    for metric_name in fold_results[0]['metrics'].keys():
        values = [f['metrics'][metric_name] for f in fold_results]
        metrics[metric_name] = (np.mean(values), np.std(values))
    return metrics

agg_metrics = aggregate_results(results['fold_results'])

print("\n" + "="*50)
print("AGGREGATED RESULTS (Mean ± Std)")
print("="*50)
for metric_name, (mean, std) in agg_metrics.items():
    print(f"{metric_name}: {mean:.4f} ± {std:.4f}")

In [ ]:
# ==================== Confusion Matrix ====================

predictions_df = results['predictions_df']

# Aggregate all predictions
all_true = predictions_df['true_label'].values
all_pred = predictions_df['pred_label'].values

cm = confusion_matrix(all_true, all_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=config.CLASS_NAMES,
            yticklabels=config.CLASS_NAMES)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix (All Folds)')
plt.tight_layout()
plt.savefig(config.OUTPUT_DIR / "confusion_matrix.png", dpi=150)
plt.show()

# Classification report
print("\n" + "="*50)
print("CLASSIFICATION REPORT")
print("="*50)
print(classification_report(all_true, all_pred, target_names=config.CLASS_NAMES))

### 6.1 Detailed Per-Class Metrics (Sensitivity / Specificity / PPV / NPV / F1)

In [ ]:
# ==================== Detailed Per-Class Metrics ====================
from sklearn.metrics import roc_auc_score, roc_curve, average_precision_score
from sklearn.preprocessing import label_binarize

predictions_df = results['predictions_df']
all_true = predictions_df['true_label'].values.astype(int)
all_pred = predictions_df['pred_label'].values.astype(int)

# Build probability matrix
prob_cols = [f'prob_{cls.lower()}' for cls in config.CLASS_NAMES]
all_probs = predictions_df[prob_cols].values

cm = confusion_matrix(all_true, all_pred)
n_classes = len(config.CLASS_NAMES)

# --- Per-class sensitivity, specificity, PPV, NPV, F1 ---
per_class_metrics = []

for i, cls_name in enumerate(config.CLASS_NAMES):
    tp = cm[i, i]
    fn = cm[i, :].sum() - tp
    fp = cm[:, i].sum() - tp
    tn = cm.sum() - tp - fn - fp

    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    ppv = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    npv = tn / (tn + fn) if (tn + fn) > 0 else 0.0
    f1 = 2 * ppv * sensitivity / (ppv + sensitivity) if (ppv + sensitivity) > 0 else 0.0
    support = int(cm[i, :].sum())

    per_class_metrics.append({
        'Class': cls_name,
        'Support (n)': support,
        'Sensitivity (Recall)': round(sensitivity, 4),
        'Specificity': round(specificity, 4),
        'PPV (Precision)': round(ppv, 4),
        'NPV': round(npv, 4),
        'F1-score': round(f1, 4),
    })

metrics_table = pd.DataFrame(per_class_metrics)

print('=' * 70)
print('PER-CLASS METRICS (5-fold CV pooled)')
print('=' * 70)
display(metrics_table)

# Overall summary
overall_acc = accuracy_score(all_true, all_pred)
f1_macro_val = f1_score(all_true, all_pred, average='macro')
f1_weighted_val = f1_score(all_true, all_pred, average='weighted')
sens_macro = np.mean([m['Sensitivity (Recall)'] for m in per_class_metrics])
spec_macro = np.mean([m['Specificity'] for m in per_class_metrics])

print(f'\nOverall Accuracy:       {overall_acc:.4f}')
print(f'Macro Sensitivity:      {sens_macro:.4f}')
print(f'Macro Specificity:      {spec_macro:.4f}')
print(f'Macro F1-score:         {f1_macro_val:.4f}')
print(f'Weighted F1-score:      {f1_weighted_val:.4f}')

### 6.2 ROC Curves and AUC (One-vs-Rest)

In [ ]:
# ==================== ROC Curves (One-vs-Rest) ====================

# Binarize labels for one-vs-rest ROC
y_true_bin = label_binarize(all_true, classes=list(range(n_classes)))

fig, axes = plt.subplots(1, n_classes, figsize=(5 * n_classes, 5))

auc_scores = {}
for i, cls_name in enumerate(config.CLASS_NAMES):
    ax = axes[i]
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], all_probs[:, i])
    auc_val = roc_auc_score(y_true_bin[:, i], all_probs[:, i])
    auc_scores[cls_name] = auc_val

    ax.plot(fpr, tpr, 'b-', lw=2, label=f'AUC = {auc_val:.4f}')
    ax.plot([0, 1], [0, 1], 'r--', lw=1)
    ax.set_xlabel('False Positive Rate (1 - Specificity)')
    ax.set_ylabel('True Positive Rate (Sensitivity)')
    ax.set_title(f'{cls_name} vs Rest')
    ax.legend(loc='lower right')
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1.02])
    ax.set_aspect('equal')

# Macro-average AUC
macro_auc = roc_auc_score(y_true_bin, all_probs, average='macro', multi_class='ovr')
auc_scores['Macro Average'] = macro_auc

plt.suptitle(f'ROC Curves (One-vs-Rest)  |  Macro AUC = {macro_auc:.4f}', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(config.OUTPUT_DIR / 'roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

# Print AUC summary
print('\n=== AUC-ROC Summary (One-vs-Rest) ===')
for cls_name, auc_val in auc_scores.items():
    print(f'  {cls_name:15s}: {auc_val:.4f}')

### 6.3 Comprehensive Summary Table

In [ ]:
# ==================== Comprehensive Summary ====================

# Combine all metrics into one table
summary_rows = []
for i, cls_name in enumerate(config.CLASS_NAMES):
    m = per_class_metrics[i]
    row = {
        'Class': cls_name,
        'n': m['Support (n)'],
        'Sensitivity': m['Sensitivity (Recall)'],
        'Specificity': m['Specificity'],
        'PPV': m['PPV (Precision)'],
        'NPV': m['NPV'],
        'F1-score': m['F1-score'],
        'AUC-ROC': round(auc_scores[cls_name], 4),
    }
    summary_rows.append(row)

# Add macro average row
summary_rows.append({
    'Class': 'Macro Avg',
    'n': sum(m['Support (n)'] for m in per_class_metrics),
    'Sensitivity': round(sens_macro, 4),
    'Specificity': round(spec_macro, 4),
    'PPV': round(np.mean([m['PPV (Precision)'] for m in per_class_metrics]), 4),
    'NPV': round(np.mean([m['NPV'] for m in per_class_metrics]), 4),
    'F1-score': round(f1_macro_val, 4),
    'AUC-ROC': round(macro_auc, 4),
})

summary_df_full = pd.DataFrame(summary_rows)

print('=' * 80)
print('COMPREHENSIVE EVALUATION SUMMARY (5-fold CV pooled)')
print(f'Overall Accuracy: {overall_acc:.4f}')
print('=' * 80)
display(summary_df_full)

# Save to Excel (updated with all metrics)
with pd.ExcelWriter(config.OUTPUT_DIR / 'evaluation_results.xlsx') as writer:
    summary_df_full.to_excel(writer, sheet_name='Summary', index=False)
    metrics_table.to_excel(writer, sheet_name='Per-Class Detail', index=False)

    # Per-fold metrics
    fold_metrics_data = []
    for fold_result in results['fold_results']:
        row = {'fold': fold_result['fold']}
        row.update(fold_result['metrics'])
        fold_metrics_data.append(row)
    pd.DataFrame(fold_metrics_data).to_excel(writer, sheet_name='Per-Fold Metrics', index=False)

    predictions_df.to_excel(writer, sheet_name='Predictions', index=False)

print(f'\nResults saved to: {config.OUTPUT_DIR / "evaluation_results.xlsx"}')

In [ ]:
# ==================== Training Curves ====================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curves
ax = axes[0]
for fold_result in results['fold_results']:
    ax.plot(fold_result['train_losses'], label=f"Fold {fold_result['fold']} Train", alpha=0.7)
    ax.plot(fold_result['val_losses'], linestyle='--', label=f"Fold {fold_result['fold']} Val", alpha=0.7)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Training and Validation Loss')
ax.legend(loc='best', fontsize=8)

# Accuracy curves
ax = axes[1]
for fold_result in results['fold_results']:
    ax.plot(fold_result['train_accs'], label=f"Fold {fold_result['fold']} Train", alpha=0.7)
    ax.plot(fold_result['val_accs'], linestyle='--', label=f"Fold {fold_result['fold']} Val", alpha=0.7)
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy')
ax.set_title('Training and Validation Accuracy')
ax.legend(loc='best', fontsize=8)

plt.tight_layout()
plt.savefig(config.OUTPUT_DIR / "training_curves.png", dpi=150)
plt.show()

In [ ]:
# ==================== Final Summary ====================

print("\n" + "="*60)
print("QUALITY CLASSIFIER TRAINING COMPLETE")
print("="*60)
print(f"\nModel: {config.MODEL_NAME}")
print(f"Classes: {config.CLASS_NAMES}")
print(f"Total images: {len(dataset_df)}")
for cls_name in config.CLASS_NAMES:
    count = len(dataset_df[dataset_df['quality'] == cls_name])
    print(f"  {cls_name}: {count}")
print(f"\nFinal Results (5-fold CV):")
print(f"  Accuracy:      {agg_metrics['accuracy'][0]:.4f} ± {agg_metrics['accuracy'][1]:.4f}")
print(f"  F1 (macro):    {agg_metrics['f1_macro'][0]:.4f} ± {agg_metrics['f1_macro'][1]:.4f}")
print(f"  F1 (weighted): {agg_metrics['f1_weighted'][0]:.4f} ± {agg_metrics['f1_weighted'][1]:.4f}")
print(f"\nOutput directory: {config.OUTPUT_DIR}")
print("="*60)

## 7. Inference Example

In [ ]:
# ==================== Inference Function ====================

def load_trained_model(model_path: Path) -> QualityClassifier:
    """Load trained model for inference."""
    model = QualityClassifier(model_name=config.MODEL_NAME, num_classes=config.NUM_CLASSES)
    model.load_state_dict(torch.load(model_path))
    model = model.to(device)
    model.eval()
    return model

def predict_quality(model: QualityClassifier, image_path: str) -> Dict:
    """Predict quality for a single image."""
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    transform = get_valid_transforms(config.IMG_SIZE)
    transformed = transform(image=image)
    image_tensor = transformed['image'].unsqueeze(0).to(device)
    
    with torch.no_grad():
        output = model(image_tensor)
        probs = F.softmax(output, dim=1).cpu().numpy()[0]
        pred_class = output.argmax(1).item()
    
    return {
        'predicted_quality': config.CLASS_NAMES[pred_class],
        'probabilities': {cls_name: float(probs[i]) for i, cls_name in enumerate(config.CLASS_NAMES)}
    }

# Example usage (using fold 1 model)
print("\n=== Inference Example ===")
model_path = config.OUTPUT_DIR / "fold_1" / "best_model.pt"
if model_path.exists():
    model = load_trained_model(model_path)
    
    # Test on a sample image
    sample_image = dataset_df.iloc[0]['image_path']
    result = predict_quality(model, sample_image)
    
    print(f"Image: {sample_image}")
    print(f"True quality: {dataset_df.iloc[0]['quality']}")
    print(f"Predicted quality: {result['predicted_quality']}")
    print(f"Probabilities: {result['probabilities']}")
else:
    print("Model not found. Run training first.")